# DQ-04 · payments.csv
Checks: null rate, duplicates, FK → orders, domain values, business rules (payment ≈ order total).

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# ── helpers ───────────────────────────────────────────────────────────────────
issues = []

def flag(label, mask_or_count, df=None, show_sample=True):
    count = int(mask_or_count.sum()) if hasattr(mask_or_count, 'sum') else int(mask_or_count)
    total = len(df) if df is not None else None
    pct   = f' ({count/total*100:.2f}%)' if total else ''
    status = '❌ ISSUE' if count > 0 else '✅ OK'
    print(f'{status}  {label}: {count:,}{pct}')
    if count > 0:
        issues.append(label)
        if show_sample and df is not None and hasattr(mask_or_count, 'sum'):
            sample = df[mask_or_count].head(3)
            print(sample.to_string(index=False))
    return count

def summary():
    print()
    if issues:
        print(f'══ {len(issues)} issue(s) found ══')
        for i in issues: print(f'  • {i}')
    else:
        print('══ All checks passed ══')


In [ ]:
pay    = pd.read_csv('payments.csv')
orders = pd.read_csv('orders.csv')
items  = pd.read_csv('order_items.csv', low_memory=False)
print(f'Shape: {pay.shape}')
pay.head(3)

## 1. Null rate

In [ ]:
null_counts = pay.isnull().sum()
print(pd.DataFrame({'null_count': null_counts, 'null_%': (null_counts/len(pay)*100).round(2)}))

## 2. 1:1 với orders (mỗi order có đúng 1 payment)

In [ ]:
flag('Duplicate order_id in payments', pay.duplicated(subset='order_id'), pay)
# Order không có payment
orders_no_pay = ~orders['order_id'].isin(pay['order_id'])
flag('Orders without payment', orders_no_pay, orders)

## 3. FK: order_id → orders

In [ ]:
valid_orders = set(orders['order_id'])
flag('order_id not in orders', ~pay['order_id'].isin(valid_orders), pay)

## 4. Domain values

In [ ]:
VALID_PAYMENT = {'credit_card','paypal','cod','apple_pay','bank_transfer'}
VALID_INST    = {1, 2, 3, 6, 12}
flag('Invalid payment_method', ~pay['payment_method'].isin(VALID_PAYMENT), pay)
flag('Invalid installments',   ~pay['installments'].isin(VALID_INST),      pay)

## 5. Business rule: payment_value > 0

In [ ]:
flag('payment_value ≤ 0', pay['payment_value'] <= 0, pay)

## 6. Business rule: payment_value ≈ net order total (qty×price − discount)

In [ ]:
net_by_order = (
    items
    .assign(net=lambda d: d['quantity']*d['unit_price'] - d['discount_amount'])
    .groupby('order_id')['net'].sum()
    .reset_index()
    .rename(columns={'net':'calc_total'})
)
df = pay.merge(net_by_order, on='order_id', how='left')
df['diff']    = (df['payment_value'] - df['calc_total']).abs()
df['diff_pct']= df['diff'] / df['calc_total'] * 100

flag('payment_value differs from net order total by > 1 VND', df['diff'] > 1.0, df)
flag('payment_value differs by > 1%',                         df['diff_pct'] > 1.0, df)
print(f"\nMax diff: {df['diff'].max():,.2f}  |  Mean diff: {df['diff'].mean():,.4f}")

## Summary

In [ ]:
summary()